# 🧠 Convolutional Neural Network (CNN) Implementation

This notebook implements a Convolutional Neural Network (CNN) in PyTorch to classify the FashionMNIST dataset from `fmnist_small.csv`.

---

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure GPU/CPU device is configured
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

### 📊 1. Load Data and Shape for CNN
Unlike an Artificial Neural Network (ANN) which accepts flat 1D vectors of shape `(784,)`, a 2D Convolutional layer (`nn.Conv2d`) requires a 4D input of shape `(batch_size, channels, height, width)`, which is `(batch_size, 1, 28, 28)` for grayscale FashionMNIST images.

We reshape the inputs in our custom PyTorch `Dataset` below.

In [ ]:
# Load FashionMNIST small dataset
df = pd.read_csv('fmnist_small.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize pixel values between [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

class FashionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        # CRITICAL FOR CNN: Reshape the 784-dimensional flat vector to (1, 28, 28) image format
        feature_image = self.features[index].view(1, 28, 28)
        return feature_image, self.labels[index]

# Create Datasets & DataLoaders
train_dataset = FashionDataset(X_train, y_train)
test_dataset = FashionDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples and {len(test_dataset)} testing samples.")

### 🏗️ 2. Define CNN Architecture
We implement a standard 2D Convolutional Neural Network.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        # Block 1: (1, 28, 28) -> (16, 28, 28) -> MaxPool (16, 14, 14)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Block 2: (16, 14, 14) -> (32, 14, 14) -> MaxPool (32, 7, 7)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Classification Head
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = out.view(out.size(0), -1) # Flatten
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

model = FashionCNN().to(device)
print(model)

### 📈 3. Training the CNN

In [ ]:
epochs = 15
learning_rate = 0.001

loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = loss_func(predictions, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

### 🎯 4. Evaluation

In [ ]:
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
        
accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")